# Stack with simple average meta (log returns)
Same as 98a but **targets are log returns** (return = log(close/close_prev)); base models (LightGBM + TCN + Ridge + EWM) predict log returns; blend = simple average; price = p0 * exp(cumsum(log_returns)). Ridge base = ridge_core; TCN from 07-tcn-pool.

In [17]:
import sys
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
import lightgbm as lgb

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    build_pooled_train_stack,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    fetch_cnn_fear_greed_index,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_STACK,
    ARTIFACTS_DIR,
    TICKERS,
)

import json
# Load best params from 05/06 random search if available (run 05 and 06 first to generate artifacts)
_lgb_path = ARTIFACTS_DIR / "best_lgb_params.json"
if _lgb_path.exists():
    with open(_lgb_path) as f:
        LGB_PARAMS = json.load(f)
    LGB_PARAMS.setdefault("random_state", 42)
    LGB_PARAMS.setdefault("verbosity", -1)
else:
    LGB_PARAMS = dict(n_estimators=100, max_depth=4, learning_rate=0.01, random_state=42, verbosity=-1)
_ridge_path = ARTIFACTS_DIR / "best_ridge_core_params.json"
if _ridge_path.exists():
    with open(_ridge_path) as f:
        _ridge = json.load(f)
    RIDGE_ALPHA = _ridge.get("alpha", 1.0)
else:
    RIDGE_ALPHA = 1.0
EWM_SPAN = 7

import ast
# TCN base: load best params from 07-tcn-pool random search
_tcn_path = ARTIFACTS_DIR / "best_tcn_params.parquet"
if _tcn_path.exists():
    try:
        _tcn_df = pd.read_parquet(_tcn_path)
        if not _tcn_df.empty:
            r = _tcn_df.iloc[0]
            TCN_PARAMS = {"seq_len": int(r["seq_len"]), "epochs": int(r["epochs"]), "filters": int(r["filters"]), "kernel_size": int(r["kernel_size"]), "dilations": ast.literal_eval(r["dilations_str"]), "batch_size": 256}
            SEQ_LEN = TCN_PARAMS["seq_len"]
        else:
            TCN_PARAMS = {"seq_len": 30, "epochs": 40, "filters": 64, "kernel_size": 3, "dilations": [1, 2, 4, 8], "batch_size": 256}
            SEQ_LEN = 30
    except Exception:
        TCN_PARAMS = {"seq_len": 30, "epochs": 40, "filters": 64, "kernel_size": 3, "dilations": [1, 2, 4, 8], "batch_size": 256}
        SEQ_LEN = 30
else:
    TCN_PARAMS = {"seq_len": 30, "epochs": 40, "filters": 64, "kernel_size": 3, "dilations": [1, 2, 4, 8], "batch_size": 256}
    SEQ_LEN = 30

LAG_RETURNS = 5
RSI_PERIOD = 7
MACD_FAST, MACD_SLOW, MACD_SIGNAL = 12, 26, 9


In [18]:
def _rsi(series: pd.Series, period: int) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / np.where(avg_loss != 0, avg_loss, 1e-10)
    return 100 - (100 / (1 + rs))


def build_feature_df(grp: pd.DataFrame):
    df = grp.sort_values("timestamp").copy()
    df["close"] = df["close"].astype(float)
    df["return"] = np.log(df["close"] / df["close"].shift(1))
    for i in range(1, LAG_RETURNS + 1):
        df[f"ret_lag_{i}"] = df["return"].shift(i)
    if "volume" in df.columns:
        df["volume_lag_1"] = df["volume"].astype(float).shift(1)
    else:
        df["volume_lag_1"] = np.nan
    df["rsi"] = _rsi(df["close"], RSI_PERIOD)
    ema_fast = df["close"].ewm(span=MACD_FAST, adjust=False).mean()
    ema_slow = df["close"].ewm(span=MACD_SLOW, adjust=False).mean()
    df["macd_line"] = ema_fast - ema_slow
    df["macd_signal"] = df["macd_line"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    if "vix" in df.columns:
        vix = df["vix"].astype(np.float64)
        df["vix_sma_5"] = vix.shift(1).rolling(5).mean()
        df["vix_velocity"] = vix.diff(1)
        df["vix_momentum"] = vix - df["vix_sma_5"]
    else:
        df["vix_sma_5"] = np.nan
        df["vix_velocity"] = np.nan
        df["vix_momentum"] = np.nan
    df["month"] = pd.to_datetime(df["timestamp"]).dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    if "fear_greed" not in df.columns:
        df["fear_greed"] = 50.0
    else:
        df["fear_greed"] = df["fear_greed"].fillna(50.0)
    df["fear_greed_lag_1"] = df["fear_greed"].shift(1)
    df["fear_greed_lag_5"] = df["fear_greed"].shift(5)
    df["fear_greed_change"] = df["fear_greed_lag_1"] - df["fear_greed_lag_5"]
    for h in range(1, FORECAST_HORIZON + 1):
        df[f"target_{h}"] = df["return"].shift(-h)
    feature_cols_tcn = [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)] + ["volume_lag_1", "rsi", "macd_line", "macd_signal"]
    feature_cols_lgb = ["vix_sma_5", "vix_velocity", "vix_momentum", "month_sin", "month_cos", "fear_greed_change"]
    target_cols = [f"target_{h}" for h in range(1, FORECAST_HORIZON + 1)]
    base_cols = ["timestamp", "close", "return"] + feature_cols_tcn + feature_cols_lgb + target_cols
    out = df[[c for c in base_cols if c in df.columns]].copy()
    return out.dropna(), feature_cols_tcn, feature_cols_lgb, target_cols


def build_sequences(X: np.ndarray, y: np.ndarray, seq_len: int):
    n = len(X)
    if n < seq_len + 1:
        return None, None
    X_seq, y_seq = [], []
    for i in range(seq_len, n):
        X_seq.append(X[i - seq_len : i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

In [19]:
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv1D, Lambda
    HAS_TCN = True
except Exception:
    HAS_TCN = False

def build_tcn(seq_len: int, n_feat: int, horizon: int, filters=64, kernel_size=3, dilations=None):
    if dilations is None:
        dilations = [1, 2, 4, 8]
    model = Sequential()
    for i, d in enumerate(dilations):
        model.add(Conv1D(filters, kernel_size, dilation_rate=d, padding="causal", activation="relu",
                        input_shape=(seq_len, n_feat) if i == 0 else None))
    model.add(Conv1D(horizon, 1))
    model.add(Lambda(lambda x: x[:, -1, :]))
    model.compile(optimizer="adam", loss="mse")
    return model

def train_tcn(X_seq: np.ndarray, y_seq: np.ndarray, horizon: int, epochs=40, filters=64, kernel_size=3, dilations=None, batch_size=256):
    if not HAS_TCN or X_seq is None or len(X_seq) < 10:
        return None
    if dilations is None:
        dilations = [1, 2, 4, 8]
    seq_len, n_feat = X_seq.shape[1], X_seq.shape[2]
    model = build_tcn(seq_len, n_feat, horizon, filters=filters, kernel_size=kernel_size, dilations=dilations)
    model.fit(X_seq, y_seq, epochs=epochs, batch_size=batch_size, verbose=0)
    return model

In [20]:
def train_global_stack_simple_avg(stacked: pd.DataFrame, horizon: int):
    """Train LightGBM + TCN + Ridge once on full pooled data (targets = log returns). EWM at predict time. Returns dict for predict_stack_simple_avg."""
    pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
    if pooled.empty:
        return None
    feat_dfs = []
    for sym in pooled["symbol"].unique():
        grp = pooled[pooled["symbol"] == sym].copy()
        try:
            feat_df, feature_cols_tcn, feature_cols_lgb, target_cols = build_feature_df(grp)
        except Exception:
            continue
        if len(feat_df) < MIN_TRAIN_STACK + horizon:
            continue
        feat_df["_symbol_"] = sym
        feat_dfs.append(feat_df)
    if not feat_dfs:
        return None
    pooled_feat = pd.concat(feat_dfs, ignore_index=True)
    feature_cols_ridge = [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)]  # ridge_core: return lags only
    y_full = pooled_feat[target_cols].values.astype(np.float32)
    scaler_lgb = StandardScaler().fit(pooled_feat[feature_cols_lgb].values.astype(np.float32))
    scaler_tcn = StandardScaler().fit(pooled_feat[feature_cols_tcn].values.astype(np.float32))
    scaler_ridge = StandardScaler().fit(pooled_feat[feature_cols_ridge].values.astype(np.float64))
    X_lgb_full_s = scaler_lgb.transform(pooled_feat[feature_cols_lgb].values.astype(np.float32))
    X_tcn_full_s = scaler_tcn.transform(pooled_feat[feature_cols_tcn].values.astype(np.float32))
    X_ridge_full_s = scaler_ridge.transform(pooled_feat[feature_cols_ridge].values.astype(np.float64))
    lgb_multi = MultiOutputRegressor(lgb.LGBMRegressor(**LGB_PARAMS))
    lgb_multi.fit(X_lgb_full_s, y_full)
    X_seq_list, y_seq_list = [], []
    for sym in pooled_feat["_symbol_"].unique():
        sym_mask = pooled_feat["_symbol_"] == sym
        X_s = X_tcn_full_s[sym_mask]
        y_s = y_full[sym_mask]
        X_seq, y_seq = build_sequences(X_s, y_s, SEQ_LEN)
        if X_seq is not None and len(X_seq) >= 10:
            X_seq_list.append(X_seq)
            y_seq_list.append(y_seq)
    tcn_model = train_tcn(np.concatenate(X_seq_list, axis=0), np.concatenate(y_seq_list, axis=0), horizon, epochs=TCN_PARAMS["epochs"], filters=TCN_PARAMS["filters"], kernel_size=TCN_PARAMS["kernel_size"], dilations=TCN_PARAMS["dilations"], batch_size=TCN_PARAMS["batch_size"]) if X_seq_list else None
    ridge_multi = MultiOutputRegressor(Ridge(alpha=RIDGE_ALPHA, random_state=42))
    ridge_multi.fit(X_ridge_full_s, y_full.astype(np.float64))
    return {
        "scaler_lgb": scaler_lgb, "scaler_tcn": scaler_tcn, "scaler_ridge": scaler_ridge,
        "lgb_multi": lgb_multi, "tcn_model": tcn_model, "ridge_multi": ridge_multi,
        "feature_cols_tcn": feature_cols_tcn, "feature_cols_lgb": feature_cols_lgb, "feature_cols_ridge": feature_cols_ridge,
        "target_cols": target_cols, "seq_len": SEQ_LEN,
    }


def predict_stack_simple_avg(context_df: pd.DataFrame, horizon: int, global_stack: dict) -> list:
    """Predict price steps: average of LGB, TCN, Ridge, EWM log-return predictions; price = p0 * exp(cumsum(log_ret))."""
    if global_stack is None or global_stack.get("lgb_multi") is None:
        return []
    try:
        feat_df, feature_cols_tcn, feature_cols_lgb, _ = build_feature_df(context_df)
    except Exception:
        return []
    seq_len = int(global_stack.get("seq_len", 30))
    if len(feat_df) < seq_len + 1:
        return []
    feature_cols_ridge = global_stack.get("feature_cols_ridge", [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)])
    scaler_lgb = global_stack.get("scaler_lgb")
    scaler_tcn = global_stack.get("scaler_tcn")
    scaler_ridge = global_stack.get("scaler_ridge")
    if scaler_lgb is None or scaler_tcn is None or scaler_ridge is None:
        return []
    lgb_multi = global_stack["lgb_multi"]
    tcn_model = global_stack["tcn_model"]
    ridge_multi = global_stack["ridge_multi"]
    EWM_SPAN = global_stack.get("EWM_SPAN", 7)
    X_lgb_s = scaler_lgb.transform(feat_df[feature_cols_lgb].values.astype(np.float32))
    X_tcn_s = scaler_tcn.transform(feat_df[feature_cols_tcn].values.astype(np.float32))
    X_ridge_s = scaler_ridge.transform(feat_df[feature_cols_ridge].values.astype(np.float64))
    last_idx = len(feat_df) - 1
    lgb_21 = lgb_multi.predict(X_lgb_s[last_idx : last_idx + 1]).ravel()
    ridge_21 = ridge_multi.predict(X_ridge_s[last_idx : last_idx + 1]).ravel()
    ret_series = feat_df["return"].values.astype(np.float64)
    if len(ret_series) >= EWM_SPAN:
        ewm_val = float(pd.Series(ret_series).ewm(span=EWM_SPAN).mean().iloc[-1])
        ewm_21 = np.full(horizon, ewm_val, dtype=np.float32)
    else:
        ewm_21 = np.full(horizon, float(np.nanmean(ret_series)) if len(ret_series) else 0.0, dtype=np.float32)
    if tcn_model is not None and last_idx >= seq_len:
        seq = X_tcn_s[last_idx - seq_len : last_idx].reshape(1, seq_len, -1)
        tcn_21 = tcn_model.predict(seq, verbose=0).ravel()
    else:
        tcn_21 = lgb_21
    final_log_returns = (np.array(lgb_21) + np.array(tcn_21) + np.array(ridge_21) + np.array(ewm_21)) / 4.0
    p0 = float(context_df["close"].iloc[-1])
    prices = p0 * np.exp(np.cumsum(final_log_returns))
    return [float(p) for p in prices[:horizon]]

In [21]:
stacked = load_pool_data(with_vix=True, with_volume=True)
symbol_start = pd.to_datetime(stacked["timestamp"]).min().strftime("%Y-%m-%d")
fear_greed_df = fetch_cnn_fear_greed_index(start_date=symbol_start)
if not fear_greed_df.empty:
    stacked["date"] = pd.to_datetime(stacked["timestamp"]).dt.normalize()
    fear_greed_df["date"] = pd.to_datetime(fear_greed_df["timestamp"]).dt.normalize()
    stacked = stacked.merge(fear_greed_df[["date", "fear_greed"]], on="date", how="left")
    stacked["fear_greed"] = stacked["fear_greed"].ffill().bfill()
    stacked = stacked.drop(columns=["date"])
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close,volume,vix,fear_greed
0,2021-03-01,AAPL,127.790001,116307900,23.350000,58.52
1,2021-03-02,AAPL,125.120003,102260900,24.100000,50.32
2,2021-03-03,AAPL,122.059998,112966300,26.670000,43.84
3,2021-03-04,AAPL,120.129997,178155000,28.570000,37.80
4,2021-03-05,AAPL,121.419998,153766600,24.660000,41.52
5,2021-03-08,AAPL,116.360001,154376600,25.469999,39.08
6,2021-03-09,AAPL,121.089996,129525800,24.030001,43.36
7,2021-03-10,AAPL,119.980003,111943300,22.559999,45.56
8,2021-03-11,AAPL,121.959999,103026500,21.910000,50.48
9,2021-03-12,AAPL,121.029999,88105100,20.690001,53.72


In [22]:
# Train base models once on full pooled data (no OOF, no meta-learner)
global_stack = train_global_stack_simple_avg(stacked, FORECAST_HORIZON)
pooled_train = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
print("Simple-avg stack (log-return) trained on", len(pooled_train), "pooled train rows.")

c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Simple-avg stack (log-return) trained on 11960 pooled train rows.


In [23]:
# Backtest: same 60-day test window, 7-day horizon, step 7 days
model_name = "stack_simple_avg_logret"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym].copy()
    if grp.empty:
        continue
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    prices = grp.set_index("timestamp")["close"].astype(float).dropna()
    n = len(prices)
    if n < TEST_SIZE + MIN_TRAIN_STACK:
        continue
    split_idx = n - TEST_SIZE
    test_index = prices.index[split_idx:]
    test_values = prices.values[split_idx:]
    preds = []
    window_ix = 0
    start = 0
    while start + FORECAST_HORIZON <= TEST_SIZE:
        context_cols = ["timestamp", "close", "vix", "symbol"] + [c for c in ["volume", "fear_greed"] if c in grp.columns]
        context_df = grp.iloc[: split_idx + start][context_cols].copy()
        if len(context_df) < MIN_TRAIN_STACK:
            start += ROLLING_STEP
            continue
        price_list = predict_stack_simple_avg(context_df, FORECAST_HORIZON, global_stack)
        if not price_list or len(price_list) < FORECAST_HORIZON:
            start += ROLLING_STEP
            window_ix += 1
            continue
        for h in range(FORECAST_HORIZON):
            idx = start + h
            ts = test_index[idx]
            y_true = float(test_values[idx])
            y_pred = float(price_list[h])
            preds.append({"timestamp": ts, "y_true": y_true, "y_pred": y_pred, "window_ix": window_ix})
        window_ix += 1
        start += ROLLING_STEP
    if preds:
        pred_df = pd.DataFrame(preds)
        pred_df["symbol"] = sym
        all_preds.append(pred_df)

pred_stack = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_stack.groupby("symbol").size() if not pred_stack.empty else "No predictions.")
pred_stack.head()

symbol
AAPL     56
AMZN     56
GOOGL    56
JNJ      56
JPM      56
MSFT     56
NVDA     56
SPY      56
WMT      56
XOM      56
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-02,286.190002,283.228588,0,AAPL
1,2025-12-03,284.149994,283.307471,0,AAPL
2,2025-12-04,280.700012,283.541576,0,AAPL
3,2025-12-05,278.779999,283.660958,0,AAPL
4,2025-12-08,277.890015,283.615946,0,AAPL


In [24]:
metrics_rows = []
for sym in pred_stack["symbol"].unique():
    sub = pred_stack[pred_stack["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_stack)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_stack_simple_avg_logret_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_stack_simple_avg_logret_pool.parquet")

                      model   symbol        MAE       RMSE    MAPE_%
0   stack_simple_avg_logret     AAPL   7.655628   8.512078  2.899336
1   stack_simple_avg_logret     MSFT  10.344584  11.220026  2.309420
2   stack_simple_avg_logret    GOOGL   7.928617   9.019627  2.507386
3   stack_simple_avg_logret     AMZN   8.400785   9.302757  3.753021
4   stack_simple_avg_logret      JPM   7.612073   8.354133  2.431189
5   stack_simple_avg_logret      JNJ   3.897392   4.309447  1.767673
6   stack_simple_avg_logret      WMT   2.754243   3.022948  2.290264
7   stack_simple_avg_logret      SPY   6.722167   7.515488  0.981791
8   stack_simple_avg_logret      XOM   4.316386   4.652756  3.157307
9   stack_simple_avg_logret     NVDA   4.396029   5.125218  2.407750
10  stack_simple_avg_logret  overall   6.402790   8.381850  2.450514
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_stack_simple_avg_logret_pool.parquet
